In [2]:
# ==================================================
# JOB MARKET INTELLIGENCE PROJECT
# NOTEBOOK 01 : DATA PREPARATION & FEATURE ENGINEERING
# ==================================================

# ==================================================
# 1. IMPORT LIBRARIES
# ==================================================

import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

# ==================================================
# 2. LOAD DATASETS
# ==================================================

clean_jobs = pd.read_csv("clean_jobs.csv")
data_analyst = pd.read_csv("DataAnalyst.csv")
naukri = pd.read_csv("naukri_data_scientist.csv")
data_science = pd.read_csv("Data_Science_jobs.csv")

# ==================================================
# 3. DATA AUDIT
# ==================================================

datasets = {
    "clean_jobs": clean_jobs,
    "data_analyst": data_analyst,
    "naukri": naukri,
    "data_science": data_science
}

print("=" * 70)
print("DATA AUDIT")
print("=" * 70)

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(f"DATASET : {name.upper()}")
    print("=" * 60)

    print("Shape :", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicates:")
    print(df.duplicated().sum())

# ==================================================
# 4. DATA CLEANING
# ==================================================

# ---------- CLEAN JOBS ----------

clean_jobs.drop(
    columns=['work_type', 'employment_type'],
    inplace=True,
    errors='ignore'
)

clean_jobs['source'] = clean_jobs['source'].fillna('Unknown')
clean_jobs['company'] = clean_jobs['company'].fillna('Unknown')

clean_jobs['date_posted'] = pd.to_datetime(
    clean_jobs['date_posted'],
    errors='coerce'
)

clean_jobs.dropna(
    subset=['description'],
    inplace=True
)

# ---------- DATA ANALYST ----------

data_analyst.drop(
    columns=['Unnamed: 0'],
    inplace=True,
    errors='ignore'
)

data_analyst.dropna(
    subset=['Company Name'],
    inplace=True
)

# ---------- NAUKRI ----------

naukri.drop_duplicates(inplace=True)

cols_to_drop = [
    'brandingTags/0/id',
    'brandingTags/0/label',
    'clientCareersUrl',
    'clientGroupId',
    'clientHeadline',
    'clientLogo',
    'consultant',
    'exclusive',
    'hideClientName',
    'companyJobsUrl',
    'jdURL',
    'logoPath',
    'logoPathV3',
    'footerPlaceholderColor',
    'footerPlaceholderLabel'
]

naukri.drop(
    columns=cols_to_drop,
    inplace=True,
    errors='ignore'
)

naukri['createdDate'] = pd.to_datetime(
    naukri['createdDate'],
    errors='coerce'
)

# ==================================================
# 5. STANDARDIZE COLUMN NAMES
# ==================================================

clean_jobs.rename(columns={
    'title': 'job_title'
}, inplace=True)

data_analyst.rename(columns={
    'Job Title': 'job_title',
    'Company Name': 'company',
    'Location': 'location',
    'Job Description': 'description',
    'Salary Estimate': 'salary',
    'Rating': 'rating',
    'Industry': 'industry'
}, inplace=True)

naukri.rename(columns={
    'title': 'job_title',
    'companyName': 'company',
    'jobDescription': 'description',
    'createdDate': 'date_posted',
    'ambitionBoxData/AggregateRating': 'rating'
}, inplace=True)

data_science.rename(columns={
    'title': 'job_title',
    'company': 'company',
    'location': 'location',
    'salary': 'salary',
    'posted_date': 'date_posted',
    'source': 'source'
}, inplace=True)

# ==================================================
# 6. CREATE FINAL DATASET VERSIONS
# ==================================================

clean_jobs_final = clean_jobs[
    [
        'job_title',
        'company',
        'location',
        'description',
        'source',
        'date_posted'
    ]
].copy()

data_analyst_final = data_analyst[
    [
        'job_title',
        'company',
        'location',
        'description',
        'salary',
        'rating',
        'industry'
    ]
].copy()

data_analyst_final['source'] = 'Glassdoor'

naukri_final = naukri[
    [
        'job_title',
        'company',
        'location',
        'description',
        'salary',
        'rating',
        'experience',
        'date_posted',
        'tagsAndSkills'
    ]
].copy()

naukri_final['source'] = 'Naukri'

data_science_final = data_science.copy()

data_science_final['description'] = np.nan
data_science_final['rating'] = np.nan
data_science_final['industry'] = np.nan
data_science_final['experience'] = np.nan
data_science_final['tagsAndSkills'] = np.nan

# ==================================================
# 7. CREATE MASTER SCHEMA
# ==================================================

master_columns = [
    'job_title',
    'company',
    'location',
    'description',
    'salary',
    'rating',
    'industry',
    'experience',
    'date_posted',
    'tagsAndSkills',
    'source'
]

all_datasets = [
    clean_jobs_final,
    data_analyst_final,
    naukri_final,
    data_science_final
]

for df in all_datasets:

    for col in master_columns:

        if col not in df.columns:
            df[col] = np.nan

# reorder

clean_jobs_final = clean_jobs_final[master_columns]
data_analyst_final = data_analyst_final[master_columns]
naukri_final = naukri_final[master_columns]
data_science_final = data_science_final[master_columns]

# ==================================================
# 8. MERGE DATASETS
# ==================================================

master_df = pd.concat(
    [
        clean_jobs_final,
        data_analyst_final,
        naukri_final,
        data_science_final
    ],
    ignore_index=True
)

print("\nMASTER DATASET SHAPE:")
print(master_df.shape)

# ==================================================
# 9. HANDLE MISSING VALUES
# ==================================================

master_df['company'] = master_df['company'].fillna('Unknown')
master_df['location'] = master_df['location'].fillna('Unknown')
master_df['source'] = master_df['source'].fillna('Unknown')

# ==================================================
# 10. FEATURE ENGINEERING
# ==================================================

master_df['text_data'] = (
    master_df['description'].fillna('') +
    ' ' +
    master_df['tagsAndSkills'].fillna('')
)

# ==================================================
# 11. SKILL EXTRACTION
# ==================================================

skills = [
    'python',
    'sql',
    'excel',
    'power bi',
    'tableau',
    'aws',
    'machine learning',
    'statistics',
    'pandas',
    'numpy',
    'r',
    'spark',
    'hadoop',
    'azure',
    'tensorflow',
    'git',
    'etl',
    'data visualization'
]

for skill in skills:

    pattern = rf'\b{re.escape(skill)}\b'

    master_df[skill] = (
        master_df['text_data']
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
        .astype(int)
    )

# ==================================================
# 12. ROLE CLASSIFICATION
# ==================================================

master_df['role_type'] = 'Other'

master_df.loc[
    master_df['job_title']
    .str.contains(
        'data analyst',
        case=False,
        na=False
    ),
    'role_type'
] = 'Data Analyst'

master_df.loc[
    master_df['job_title']
    .str.contains(
        'data scientist',
        case=False,
        na=False
    ),
    'role_type'
] = 'Data Scientist'

# ==================================================
# 13. SKILL DEMAND ANALYSIS
# ==================================================

skill_counts = (
    master_df[skills]
    .sum()
    .sort_values(ascending=False)
)

print("\nTOP SKILLS")
print(skill_counts)

skill_percent = (
    master_df[skills]
    .mean() * 100
).sort_values(ascending=False)

print("\nSKILL DEMAND (%)")
print(skill_percent.round(2))

# ==================================================
# 14. ROLE-WISE SKILL ANALYSIS
# ==================================================

role_skill = (
    master_df
    .groupby('role_type')[skills]
    .mean() * 100
)

print("\nROLE SKILL MATRIX")
print(role_skill.round(2))

# ==================================================
# 15. SALARY EXPLORATION
# ==================================================

print("\nSALARY SAMPLE VALUES")

salary_sample = (
    master_df['salary']
    .dropna()
    .sample(
        min(20, master_df['salary'].dropna().shape[0]),
        random_state=42
    )
)

print(salary_sample)

# ==================================================
# 16. EXPERIENCE EXPLORATION
# ==================================================

print("\nEXPERIENCE SAMPLE VALUES")

if master_df['experience'].notna().sum() > 0:

    exp_sample = (
        master_df['experience']
        .dropna()
        .sample(
            min(20, master_df['experience'].dropna().shape[0]),
            random_state=42
        )
    )

    print(exp_sample)

# ==================================================
# 17. FINAL DATASET CHECK
# ==================================================

print("\nFINAL DATASET INFO")
print(master_df.info())

print("\nFINAL SHAPE")
print(master_df.shape)

print("\nROLE DISTRIBUTION")
print(master_df['role_type'].value_counts())
# ---------------------------------------------------
import re
import numpy as np

def clean_experience(exp):

    if pd.isna(exp):
        return np.nan

    nums = re.findall(r'\d+', str(exp))

    if len(nums) >= 2:
        return (int(nums[0]) + int(nums[1])) / 2

    elif len(nums) == 1:
        return int(nums[0])

    return np.nan

master_df['experience_clean'] = master_df['experience'].apply(clean_experience)

# ==================================================
# 18. EXPORT DATASET
# ==================================================

master_df.to_csv(
    "master_jobs_dataset_v1.csv",
    index=False
)

print("\nDataset Exported Successfully")
print("File Name : master_jobs_dataset_v1.csv")


master_df['experience_clean'].describe()

master_df[['experience','experience_clean']].head(20)

master_df['experience_clean'].notna().sum()
master_df[
    master_df['experience_clean'].notna()
][['experience','experience_clean']].head(20)

master_df.to_csv(
    "job_market_dashboard1.csv",
    index=False
)

master_df.drop(columns=['date_posted'], inplace=True)
master_df.to_csv(
    "job_market_dashboard2.csv",
    index=False
)
dashboard_df = master_df[
    [
        'job_title',
        'company',
        'location',
        'role_type',
        'experience_clean',
        'python',
        'sql',
        'excel',
        'power bi',
        'tableau',
        'aws',
        'machine learning',
        'statistics',
        'pandas',
        'numpy',
        'r',
        'spark',
        'hadoop',
        'azure',
        'tensorflow',
        'git',
        'etl',
        'data visualization'
    ]
]

dashboard_df.to_csv(
    "job_market_dashboard_final.csv",
    index=False
)

master_df.to_csv(
    "job_market_dashboard.csv",
    index=False,
    encoding="utf-8-sig"
)

location_df = pd.DataFrame(
    sorted(master_df['location'].dropna().unique()),
    columns=['location']
)

location_df.to_csv("all_locations.csv", index=False)

location_df.head()

DATA AUDIT

DATASET : CLEAN_JOBS
Shape : (1048, 10)

Columns:
['id', 'title', 'company', 'location', 'link', 'source', 'date_posted', 'work_type', 'employment_type', 'description']

Missing Values:
id                    0
title                 0
company               2
location             15
link                  0
source               89
date_posted          23
work_type          1048
employment_type    1048
description           4
dtype: int64

Duplicates:
0

DATASET : DATA_ANALYST
Shape : (2253, 16)

Columns:
['Unnamed: 0', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors', 'Easy Apply']

Missing Values:
Unnamed: 0           0
Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         1
Location             0
Headquarters         0
Size                 0
Founded              0
Type of owne

,location
0,"*** *********, **"
1,"*** *******, **"
2,"*** ****, **"
3,"**** ***** *** ****è*, *********, *****"
4,"***** *******, *******, *********"
